# 2-Stage Pipeline: nnU-Net (Coarse) → BA-Net (Fine) on KiTS23

## Tổng quan kiến trúc

Đây là pipeline **2-stage** mô phỏng cách tác giả áp dụng trên bộ KIPA22, được điều chỉnh để train trên **KiTS23** (kidney + tumor + cyst):

```
CT Volume (full)
      │
      ▼
┌─────────────────────────────────────────┐
│  STAGE 1 — nnU-Net (Coarse)             │
│  ResidualEncoder + 3D UNet Decoder      │
│  Deep Supervision (4 scales)            │
│  Input: full volume patches 128³        │
│  Output: coarse segmentation mask       │
└─────────────────────────────────────────┘
      │
      ▼  (crop bounding box + margin từ coarse mask)
┌─────────────────────────────────────────┐
│  STAGE 2 — BA-Net (Fine, Boundary-Aware)│
│  ResidualEncoder + AttentionGate Decoder│
│  Dual Head: seg_head + boundary_head    │
│  Input: cropped ROI 96³                 │
│  Output: fine segmentation mask         │
└─────────────────────────────────────────┘
      │
      ▼
Final prediction (paste crop back to original space)
```

**Classes KiTS23**: 0=background, 1=kidney, 2=tumor, 3=cyst


## 1. Cài đặt môi trường

In [ ]:
!pip install -q monai[all]
!pip install -q nibabel
!pip install -q trimesh fast_simplification

In [ ]:
import os
import warnings
# Suppress MONAI/PyTorch internal deprecation warnings không ảnh hưởng kết quả
warnings.filterwarnings("ignore", message=".*non-tuple sequence.*multidimensional indexing.*")
warnings.filterwarnings("ignore", message=".*DataLoader.*num_workers.*")
warnings.filterwarnings("ignore", message=".*Num foregrounds.*")
warnings.filterwarnings("ignore", message=".*unable to generate class balanced.*")

os.environ["MONAI_USE_CUPY"] = "0"

import gc
import glob
import json
import time
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from monai.transforms import (
    Compose, LoadImaged, EnsureChannelFirstd, Orientationd,
    Spacingd, ScaleIntensityRanged, CropForegroundd, ToTensord,
    RandCropByPosNegLabeld, RandRotate90d, RandFlipd,
    RandGaussianNoised, RandShiftIntensityd, RandScaleIntensityd,
    RandAffined, RandZoomd, SpatialPadd,
)
from monai.losses import DiceCELoss
from monai.metrics import DiceMetric
from monai.networks.utils import one_hot
from monai.inferers import sliding_window_inference
from scipy.ndimage import binary_erosion, distance_transform_edt
from sklearn.model_selection import train_test_split

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


## 2. Global Config

In [ ]:
# ============================================================
# GLOBAL CONFIG
# ============================================================
CFG = {
    # ── Paths ──────────────────────────────────────────────
    # Thư mục chứa dữ liệu KiTS23 gốc (mỗi case là case_XXXXX/)
    # bên trong có imaging.nii và segmentation.nii
    "raw_data_dir"    : "/kaggle/input/kits23/dataset",
    # Thư mục lưu preprocessed .pt files
    "preprocess_dir"  : "/kaggle/working/preprocessed",
    # Thư mục lưu outputs (checkpoints, logs, plots)
    "output_dir"      : "/kaggle/working/output",  # Kaggle persist toàn bộ /kaggle/working/

    # ── Dataset ────────────────────────────────────────────
    "num_classes"     : 4,       # 0=bg, 1=kidney, 2=tumor, 3=cyst
    "in_channels"     : 1,
    "train_ratio"     : 0.70,
    "val_ratio"       : 0.15,
    "test_ratio"      : 0.15,
    "random_seed"     : 42,

    # ── Preprocessing ──────────────────────────────────────
    # Spacing chuẩn nnU-Net cho kidney CT (mm)
    "target_spacing"  : (1.5, 1.5, 2.0),
    "hu_min"          : -100,
    "hu_max"          : 400,

    # ── Stage 1 — nnU-Net ──────────────────────────────────
    "s1_patch_size"   : (128, 128, 128),
    "s1_base_features": 32,
    "s1_num_patches"  : 2,
    "s1_epochs"       : 200,
    "s1_batch_size"   : 2,
    "s1_lr"           : 1e-4,
    "s1_weight_decay" : 1e-5,
    "s1_warmup_epochs": 10,
    "s1_val_every"    : 5,
    # Deep supervision weights (coarsest → finest)
    "s1_ds_weights"   : [0.5, 0.25, 0.125, 0.0625],

    # ── Stage 2 — BA-Net ───────────────────────────────────
    "s2_patch_size"   : (96, 96, 96),
    "s2_base_features": 32,
    "s2_num_patches"  : 2,
    "s2_boundary_weight": 0.4,  # w_bnd in total_loss
    "s2_epochs"       : 200,
    "s2_batch_size"   : 2,
    "s2_lr"           : 5e-5,
    "s2_weight_decay" : 1e-5,
    "s2_warmup_epochs": 10,
    "s2_val_every"    : 5,
    # Margin (voxels) khi crop ROI từ stage 1 prediction
    "s2_crop_margin"  : 20,

    # ── Training ───────────────────────────────────────────
    "num_workers"     : 2,
    "use_amp"         : True,
    "pos_neg_ratio"   : 1,
}

for d in [CFG["preprocess_dir"], CFG["output_dir"]]:
    os.makedirs(os.path.join(d, "images"), exist_ok=True)
    os.makedirs(os.path.join(d, "labels"), exist_ok=True)
for sub in ["s1_checkpoints", "s2_checkpoints", "nifti_predictions"]:
    os.makedirs(os.path.join(CFG["output_dir"], sub), exist_ok=True)

random.seed(CFG["random_seed"])
np.random.seed(CFG["random_seed"])
torch.manual_seed(CFG["random_seed"])
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(CFG["random_seed"])

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")


## 3. Preprocessing — NIfTI → .pt

In [ ]:
# ============================================================
# PREPROCESSING PIPELINE
# Áp dụng chuẩn nnU-Net:
#   1. Reorient → RAS
#   2. Resample về spacing (1.5, 1.5, 2.0)
#   3. Clip HU [-100, 400] rồi normalize [0, 1]
#   4. CropForeground (loại bỏ air voxels)
#   5. Lưu float16 (image) và uint8 (label)
# ============================================================

train_pipeline = Compose([
    LoadImaged(keys=["image", "label"], image_only=False),
    EnsureChannelFirstd(keys=["image", "label"]),
    # labels=None: dùng meta-tensor space để xác định axis labels
    # (chuẩn MONAI >= 1.3, tránh FutureWarning về default labels thay đổi)
    Orientationd(keys=["image", "label"], axcodes="RAS", labels=None),
    Spacingd(
        keys=["image", "label"],
        pixdim=CFG["target_spacing"],
        mode=("bilinear", "nearest"),
    ),
    ScaleIntensityRanged(
        keys=["image"],
        a_min=CFG["hu_min"], a_max=CFG["hu_max"],
        b_min=0.0, b_max=1.0, clip=True,
    ),
    CropForegroundd(keys=["image", "label"], source_key="image", margin=5),
    ToTensord(keys=["image", "label"]),
])


def validate_kits23_label(label_tensor):
    """Kiểm tra label KiTS23 chỉ có values 0,1,2,3."""
    vals = set(torch.unique(label_tensor).tolist())
    valid = {0, 1, 2, 3}
    invalid = vals - valid
    if invalid:
        print(f"  [WARN] Label có giá trị không hợp lệ: {invalid}")
        return False
    return True


def preprocess_case(folder, pipeline, save_dir):
    case_id  = os.path.basename(folder)
    img_path = os.path.join(folder, "imaging.nii")
    lbl_path = os.path.join(folder, "segmentation.nii")

    # KiTS23 có thể dùng .nii.gz
    if not os.path.exists(img_path):
        img_path = img_path + ".gz"
    if not os.path.exists(lbl_path):
        lbl_path = lbl_path + ".gz"

    if not os.path.exists(img_path):
        print(f"[SKIP] {case_id}: không tìm thấy imaging")
        return False

    has_label = os.path.exists(lbl_path)

    try:
        data_dict = {"image": img_path}
        if has_label:
            data_dict["label"] = lbl_path

        processed = pipeline(data_dict)

        img_t = processed["image"].to(torch.float16)
        torch.save(img_t, os.path.join(save_dir, "images", f"{case_id}.pt"))

        if has_label:
            lbl_t = processed["label"].to(torch.uint8)
            validate_kits23_label(lbl_t)
            torch.save(lbl_t, os.path.join(save_dir, "labels", f"{case_id}.pt"))
            print(f"[OK] {case_id} | shape: {img_t.shape} | labels: {torch.unique(lbl_t).tolist()}")
        else:
            print(f"[OK] {case_id} | shape: {img_t.shape} | inference only")

        return True

    except Exception as e:
        import traceback
        print(f"[ERR] {case_id}: {e}")
        traceback.print_exc()
        return False
    finally:
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


# ---- Chạy preprocessing ----
folders = sorted(glob.glob(os.path.join(CFG["raw_data_dir"], "case_*")))
print(f"Tổng số cases (toàn bộ): {len(folders)}")

# ── THỬ NGHIỆM: chỉ lấy N case đầu ──────────────────────────
# Đổi thành None để chạy toàn bộ dataset
DEBUG_N_CASES = 10
if DEBUG_N_CASES is not None:
    folders = folders[:DEBUG_N_CASES]
    print(f"[DEBUG] Chỉ chạy {len(folders)} case đầu")
# ─────────────────────────────────────────────────────────────

success, failed = 0, 0
for i, folder in enumerate(folders, 1):
    print(f"[{i}/{len(folders)}] {os.path.basename(folder)}...")
    if preprocess_case(folder, train_pipeline, CFG["preprocess_dir"]):
        success += 1
    else:
        failed += 1

print(f"\nHoàn thành: {success} OK / {failed} lỗi")


## 4. Dataset & DataLoader

In [ ]:
# ============================================================
# DATA SPLIT
# ============================================================

def build_splits(preprocess_dir, train_ratio, val_ratio, seed):
    image_paths = sorted(glob.glob(os.path.join(preprocess_dir, "images", "*.pt")))
    label_paths = sorted(glob.glob(os.path.join(preprocess_dir, "labels", "*.pt")))
    assert len(image_paths) == len(label_paths), "Mismatch images/labels!"

    img_ids = [os.path.splitext(os.path.basename(p))[0] for p in image_paths]
    lbl_ids = [os.path.splitext(os.path.basename(p))[0] for p in label_paths]
    assert img_ids == lbl_ids, "Case IDs không khớp!"

    all_cases = [{"image": img, "label": lbl} for img, lbl in zip(image_paths, label_paths)]
    n = len(all_cases)

    train_cases, temp = train_test_split(all_cases, train_size=train_ratio, random_state=seed)
    val_ratio_adj     = val_ratio / (1.0 - train_ratio)
    val_cases, test_cases = train_test_split(temp, train_size=val_ratio_adj, random_state=seed)

    print(f"Total: {n} | Train: {len(train_cases)} | Val: {len(val_cases)} | Test: {len(test_cases)}")

    split_record = {
        "train": [c["image"] for c in train_cases],
        "val"  : [c["image"] for c in val_cases],
        "test" : [c["image"] for c in test_cases],
    }
    with open(os.path.join(CFG["output_dir"], "data_split.json"), "w") as f:
        json.dump(split_record, f, indent=2)

    return train_cases, val_cases, test_cases


train_files, val_files, test_files = build_splits(
    CFG["preprocess_dir"], CFG["train_ratio"], CFG["val_ratio"], CFG["random_seed"]
)


In [ ]:
# ============================================================
# TRANSFORMS
# ============================================================

def make_train_transforms(patch_size, num_patches):
    return Compose([
        # Pad volume nếu bất kỳ chiều nào nhỏ hơn patch_size
        # (constant pad = 0 tương đương background HU sau normalize)
        SpatialPadd(keys=["image", "label"], spatial_size=patch_size, mode="constant"),
        RandCropByPosNegLabeld(
            keys=["image", "label"], label_key="label",
            spatial_size=patch_size,
            pos=CFG["pos_neg_ratio"], neg=CFG["pos_neg_ratio"],
            num_samples=num_patches,
            image_key="image", image_threshold=0.0,
            allow_smaller=True,
        ),
        RandFlipd(keys=["image", "label"], spatial_axis=0, prob=0.5),
        RandFlipd(keys=["image", "label"], spatial_axis=1, prob=0.5),
        RandFlipd(keys=["image", "label"], spatial_axis=2, prob=0.5),
        RandRotate90d(keys=["image", "label"], prob=0.5, max_k=3),
        RandAffined(
            keys=["image", "label"],
            prob=0.3,
            rotate_range=(0.15, 0.15, 0.15),
            scale_range=(0.1, 0.1, 0.1),
            mode=("bilinear", "nearest"),
            padding_mode="border",
        ),
        RandGaussianNoised(keys=["image"], prob=0.2, std=0.05),
        RandShiftIntensityd(keys=["image"], offsets=0.10, prob=0.5),
        RandScaleIntensityd(keys=["image"], factors=0.10, prob=0.5),
    ])


def make_val_transforms(patch_size):
    return Compose([
        SpatialPadd(keys=["image", "label"], spatial_size=patch_size, mode="constant"),
        RandCropByPosNegLabeld(
            keys=["image", "label"], label_key="label",
            spatial_size=patch_size,
            pos=1, neg=1, num_samples=2,
            image_key="image", image_threshold=0.0,
            allow_smaller=True,
        ),
    ])


# ============================================================
# DATASET — dùng chung cho cả Stage 1 và Stage 2
# ============================================================

class KiTS23Dataset(Dataset):
    """
    Load preprocessed .pt tensors.
    mode='stage1' : trả về toàn bộ volume (patch được cắt bởi transforms)
    mode='stage2' : trả về ROI crop (được cắt dựa trên stage1_pred_dir)
    """
    def __init__(self, data_list, transforms=None, mode="stage1", stage1_pred_dir=None,
                 crop_margin=20):
        self.data_list       = data_list
        self.transforms      = transforms
        self.mode            = mode
        self.stage1_pred_dir = stage1_pred_dir
        self.crop_margin     = crop_margin

    def __len__(self):
        return len(self.data_list)

    def __getitem__(self, idx):
        entry  = self.data_list[idx]
        img    = torch.load(entry["image"], weights_only=False).float()  # [1, D, H, W]
        lbl    = torch.load(entry["label"], weights_only=False).long()   # [1, D, H, W]

        if self.mode == "stage2" and self.stage1_pred_dir is not None:
            case_id   = os.path.splitext(os.path.basename(entry["image"]))[0]
            pred_path = os.path.join(self.stage1_pred_dir, f"{case_id}_s1pred.pt")

            if os.path.exists(pred_path):
                s1_pred = torch.load(pred_path, weights_only=False)
                s1_has_fg = (s1_pred > 0).any().item()
                if s1_has_fg:
                    img, lbl = self._crop_roi(img, lbl, s1_pred)
                else:
                    img, lbl = self._crop_roi(img, lbl, lbl)
            else:
                img, lbl = self._crop_roi(img, lbl, lbl)

            # Safety: nếu sau crop vẫn không có foreground trong label
            # → giữ nguyên full volume để RandCropByPosNegLabeld tự xử lý
            if (lbl > 0).sum() == 0:
                img = torch.load(entry["image"], weights_only=False).float()
                lbl = torch.load(entry["label"], weights_only=False).long()

        data = {"image": img, "label": lbl.float()}

        if self.transforms:
            data = self.transforms(data)

        if isinstance(data, list):
            for d in data:
                d["label"] = d["label"].long()
        else:
            data["label"] = data["label"].long()

        return data

    def _crop_roi(self, img, lbl, mask):
        """
        Crop bounding box của foreground trong mask + margin.
        mask: [1, D, H, W] — foreground = mask > 0

        Nếu mask không có foreground (S1 pred toàn background),
        caller nên dùng GT label làm mask thay thế.
        """
        fg = (mask.squeeze(0) > 0)  # [D, H, W]
        if fg.sum() == 0:
            # Trả về nguyên volume — SpatialPadd + RandCrop sẽ xử lý
            return img, lbl

        coords = torch.nonzero(fg, as_tuple=False)  # [N, 3]
        m = self.crop_margin
        D, H, W = img.shape[1], img.shape[2], img.shape[3]

        d0 = max(coords[:, 0].min().item() - m, 0)
        d1 = min(coords[:, 0].max().item() + m + 1, D)
        h0 = max(coords[:, 1].min().item() - m, 0)
        h1 = min(coords[:, 1].max().item() + m + 1, H)
        w0 = max(coords[:, 2].min().item() - m, 0)
        w1 = min(coords[:, 2].max().item() + m + 1, W)

        img_crop = img[:, d0:d1, h0:h1, w0:w1]
        lbl_crop = lbl[:, d0:d1, h0:h1, w0:w1]

        # Kiểm tra crop có chứa foreground từ GT không
        # (quan trọng khi crop theo S1 pred có thể lệch so với GT)
        if (lbl_crop > 0).sum() == 0:
            # Crop lệch hoàn toàn → fallback về full volume
            return img, lbl

        return img_crop, lbl_crop


def collate_patches(batch):
    images, labels = [], []
    for sample in batch:
        if isinstance(sample, list):
            for patch in sample:
                images.append(patch["image"])
                labels.append(patch["label"])
        else:
            images.append(sample["image"])
            labels.append(sample["label"])
    return {"image": torch.stack(images), "label": torch.stack(labels)}


print("Dataset và transforms sẵn sàng.")


## 5. Model Architecture

### 5a. Stage 1 — nnU-Net (Residual Encoder + Deep Supervision)

Implement theo kiến trúc nnU-Net v2 plain 3D:
- **Residual Encoder**: mỗi block là double conv + skip connection 1×1
- **Decoder**: ConvTranspose3d + skip concat
- **Deep Supervision**: 4 heads ở các scale khác nhau
- **Instance Norm + LeakyReLU** (theo chuẩn nnU-Net)


In [ ]:
# ============================================================
# STAGE 1: nnU-Net Architecture
# ============================================================

class ResBlock(nn.Module):
    """Residual block theo nnU-Net: 2 conv + skip nếu channel thay đổi."""
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.conv1 = nn.Sequential(
            nn.Conv3d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False),
            nn.InstanceNorm3d(out_ch, affine=True),
            nn.LeakyReLU(0.01, inplace=True),
        )
        self.conv2 = nn.Sequential(
            nn.Conv3d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.InstanceNorm3d(out_ch, affine=True),
            nn.LeakyReLU(0.01, inplace=True),
        )
        self.skip = (
            nn.Sequential(
                nn.Conv3d(in_ch, out_ch, 1, stride=stride, bias=False),
                nn.InstanceNorm3d(out_ch, affine=True),
            )
            if (in_ch != out_ch or stride != 1) else nn.Identity()
        )

    def forward(self, x):
        return self.conv2(self.conv1(x)) + self.skip(x)


class nnUNet3D(nn.Module):
    """
    3D nnU-Net với Residual Encoder và Deep Supervision.

    Encoder: 5 tầng (1 stem + 4 downsample bằng stride=2)
    Bottleneck: ResBlock ở độ phân giải thấp nhất
    Decoder: 4 tầng upsample + skip concat + ResBlock
    Deep Supervision: output logit tại mỗi tầng decoder
        → Loss được weighted sum (coarse-to-fine)
    """

    def __init__(self, in_channels=1, num_classes=4, base_features=32):
        super().__init__()
        f = base_features

        # ---- Encoder (stride-2 downsampling) ----
        self.stem  = ResBlock(in_channels, f)
        self.enc1  = ResBlock(f,      f * 2,  stride=2)
        self.enc2  = ResBlock(f * 2,  f * 4,  stride=2)
        self.enc3  = ResBlock(f * 4,  f * 8,  stride=2)
        self.enc4  = ResBlock(f * 8,  f * 16, stride=2)

        # ---- Bottleneck ----
        self.bottleneck = ResBlock(f * 16, f * 16)

        # ---- Decoder ----
        self.up4  = nn.ConvTranspose3d(f * 16, f * 8,  2, stride=2)
        self.dec4 = ResBlock(f * 16, f * 8)

        self.up3  = nn.ConvTranspose3d(f * 8,  f * 4,  2, stride=2)
        self.dec3 = ResBlock(f * 8,  f * 4)

        self.up2  = nn.ConvTranspose3d(f * 4,  f * 2,  2, stride=2)
        self.dec2 = ResBlock(f * 4,  f * 2)

        self.up1  = nn.ConvTranspose3d(f * 2,  f,      2, stride=2)
        self.dec1 = ResBlock(f * 2,  f)

        # ---- Deep Supervision Heads ----
        # Mỗi head tương ứng một tầng decoder (từ coarse đến fine)
        self.ds_head4 = nn.Conv3d(f * 8,  num_classes, 1)  # 1/8 resolution
        self.ds_head3 = nn.Conv3d(f * 4,  num_classes, 1)  # 1/4 resolution
        self.ds_head2 = nn.Conv3d(f * 2,  num_classes, 1)  # 1/2 resolution
        self.ds_head1 = nn.Conv3d(f,      num_classes, 1)  # full resolution

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, (nn.Conv3d, nn.ConvTranspose3d)):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="leaky_relu")
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x):
        # Encoder
        s0 = self.stem(x)        # [B, f,     D,   H,   W  ]
        s1 = self.enc1(s0)       # [B, f*2,   D/2, H/2, W/2]
        s2 = self.enc2(s1)       # [B, f*4,   D/4, H/4, W/4]
        s3 = self.enc3(s2)       # [B, f*8,   D/8, H/8, W/8]
        s4 = self.enc4(s3)       # [B, f*16,  D/16,H/16,W/16]

        # Bottleneck
        b  = self.bottleneck(s4)

        # Decoder + deep supervision
        d4 = self.dec4(torch.cat([self._upsample(self.up4(b), s3), s3], dim=1))
        d3 = self.dec3(torch.cat([self._upsample(self.up3(d4), s2), s2], dim=1))
        d2 = self.dec2(torch.cat([self._upsample(self.up2(d3), s1), s1], dim=1))
        d1 = self.dec1(torch.cat([self._upsample(self.up1(d2), s0), s0], dim=1))

        out4 = self.ds_head4(d4)  # coarsest
        out3 = self.ds_head3(d3)
        out2 = self.ds_head2(d2)
        out1 = self.ds_head1(d1)  # finest

        # Tại inference, chỉ dùng finest output
        if not self.training:
            return out1

        return out1, out2, out3, out4  # fine → coarse

    @staticmethod
    def _upsample(x, ref):
        """Upsample x đến kích thước của ref nếu cần."""
        if x.shape[2:] != ref.shape[2:]:
            x = F.interpolate(x, size=ref.shape[2:], mode="trilinear", align_corners=True)
        return x


# ---- Test Stage 1 ----
s1_model = nnUNet3D(
    in_channels=CFG["in_channels"],
    num_classes=CFG["num_classes"],
    base_features=CFG["s1_base_features"],
).to(DEVICE)

total = sum(p.numel() for p in s1_model.parameters())
print(f"[Stage 1] Parameters: {total:,}")

with torch.no_grad():
    dummy = torch.zeros(1, 1, 64, 64, 64).to(DEVICE)  # test với patch nhỏ
    s1_model.train()
    outs = s1_model(dummy)
    print(f"[Stage 1] Output shapes (training): {[o.shape for o in outs]}")
    s1_model.eval()
    out_eval = s1_model(dummy)
    print(f"[Stage 1] Output shape (inference): {out_eval.shape}")
del dummy; gc.collect()


### 5b. Stage 2 — BA-Net (Boundary-Aware Network)

- **Residual Encoder** (giống stage 1 nhưng nhận cropped ROI nhỏ hơn)
- **Decoder với Attention Gate** (Oktay et al., 2018)
- **Dual Head**:
  - `seg_head`: multi-class segmentation
  - `bound_head`: binary boundary detection
- **Boundary features** được inject vào decoder qua attention


In [ ]:
# ============================================================
# STAGE 2: BA-Net Architecture
# ============================================================

class AttentionGate(nn.Module):
    """
    Additive Attention Gate (Oktay et al., 2018).
    g = gating signal từ decoder path
    x = skip feature từ encoder path
    → trả về x được re-weight bởi attention map
    """
    def __init__(self, F_g, F_l, F_int):
        super().__init__()
        self.W_g = nn.Sequential(
            nn.Conv3d(F_g,  F_int, 1, bias=False),
            nn.InstanceNorm3d(F_int, affine=True),
        )
        self.W_x = nn.Sequential(
            nn.Conv3d(F_l,  F_int, 1, bias=False),
            nn.InstanceNorm3d(F_int, affine=True),
        )
        self.psi = nn.Sequential(
            nn.Conv3d(F_int, 1, 1, bias=False),
            nn.InstanceNorm3d(1, affine=True),
            nn.Sigmoid(),
        )
        self.relu = nn.ReLU(inplace=True)

    def forward(self, g, x):
        g1 = self.W_g(g)
        x1 = self.W_x(x)
        if g1.shape[2:] != x1.shape[2:]:
            g1 = F.interpolate(g1, size=x1.shape[2:], mode="trilinear", align_corners=True)
        alpha = self.psi(self.relu(g1 + x1))  # attention map [B, 1, D, H, W]
        return x * alpha


class BANet3D(nn.Module):
    """
    Boundary-Aware Network (Stage 2).

    Kiến trúc:
      - Encoder: ResBlock (giống nnU-Net) để tận dụng feature learning tốt
      - Decoder: Attention Gate + ResBlock (boundary cues hỗ trợ segmentation)
      - Seg head   : multi-class segmentation logits
      - Bound head : binary boundary logits (từ lightweight decoder branch)

    Cải tiến so với BA-Net gốc trong file cũ:
      - Encoder là ResBlock thay vì simple ConvBlock → học feature sâu hơn
      - Boundary decoder nhận skip features từ encoder (không chỉ từ bottleneck)
      - Boundary attention: attention map từ bound decoder inject vào seg decoder
    """

    def __init__(self, in_channels=1, num_classes=4, base_features=32):
        super().__init__()
        f = base_features

        # ---- Encoder ----
        self.enc1 = ResBlock(in_channels, f)
        self.enc2 = ResBlock(f,      f * 2,  stride=2)
        self.enc3 = ResBlock(f * 2,  f * 4,  stride=2)
        self.enc4 = ResBlock(f * 4,  f * 8,  stride=2)

        # ---- Bottleneck ----
        self.bottleneck = ResBlock(f * 8, f * 16, stride=2)

        # ---- Segmentation Decoder + Attention Gates ----
        self.up4  = nn.ConvTranspose3d(f * 16, f * 8,  2, stride=2)
        self.ag4  = AttentionGate(f * 8,  f * 8,  f * 4)
        self.dec4 = ResBlock(f * 16, f * 8)

        self.up3  = nn.ConvTranspose3d(f * 8,  f * 4,  2, stride=2)
        self.ag3  = AttentionGate(f * 4,  f * 4,  f * 2)
        self.dec3 = ResBlock(f * 8,  f * 4)

        self.up2  = nn.ConvTranspose3d(f * 4,  f * 2,  2, stride=2)
        self.ag2  = AttentionGate(f * 2,  f * 2,  f)
        self.dec2 = ResBlock(f * 4,  f * 2)

        self.up1  = nn.ConvTranspose3d(f * 2,  f,      2, stride=2)
        self.ag1  = AttentionGate(f,      f,      f // 2)
        self.dec1 = ResBlock(f * 2,  f)

        # ---- Segmentation Head ----
        self.seg_head = nn.Conv3d(f, num_classes, 1)

        # ---- Boundary Decoder (lightweight, chia sẻ encoder features) ----
        # Nhận skip features từ enc1, enc2 để boundary fine-grained hơn
        self.bnd_up4  = nn.ConvTranspose3d(f * 16, f * 4,  2, stride=2)
        self.bnd_dec4 = ResBlock(f * 4 + f * 8, f * 4)

        self.bnd_up3  = nn.ConvTranspose3d(f * 4,  f * 2,  2, stride=2)
        self.bnd_dec3 = ResBlock(f * 2 + f * 4, f * 2)

        self.bnd_up2  = nn.ConvTranspose3d(f * 2,  f,      2, stride=2)
        self.bnd_dec2 = ResBlock(f + f * 2, f)

        self.bnd_up1  = nn.ConvTranspose3d(f,      f,      2, stride=2)
        self.bnd_dec1 = ResBlock(f + f,     f)

        self.bound_head = nn.Conv3d(f, 1, 1)

        # ---- Boundary → Seg Attention Injection ----
        # Attention map từ boundary path được nhân vào fine decoder feature
        self.bnd_attn_gate = nn.Sequential(
            nn.Conv3d(f, 1, 1),
            nn.Sigmoid(),
        )

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, (nn.Conv3d, nn.ConvTranspose3d)):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="leaky_relu")
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    @staticmethod
    def _align(x, ref):
        if x.shape[2:] != ref.shape[2:]:
            x = F.interpolate(x, size=ref.shape[2:], mode="trilinear", align_corners=True)
        return x

    def forward(self, x):
        # Encoder
        e1 = self.enc1(x)       # [B, f,    ...]
        e2 = self.enc2(e1)      # [B, f*2,  ...]
        e3 = self.enc3(e2)      # [B, f*4,  ...]
        e4 = self.enc4(e3)      # [B, f*8,  ...]
        b  = self.bottleneck(e4) # [B, f*16, ...]

        # ─── Boundary Decoder ───────────────────────────────
        bd4 = self.bnd_dec4(torch.cat([self._align(self.bnd_up4(b), e4), e4], dim=1))
        bd3 = self.bnd_dec3(torch.cat([self._align(self.bnd_up3(bd4), e3), e3], dim=1))
        bd2 = self.bnd_dec2(torch.cat([self._align(self.bnd_up2(bd3), e2), e2], dim=1))
        bd1 = self.bnd_dec1(torch.cat([self._align(self.bnd_up1(bd2), e1), e1], dim=1))

        bound_logits = self.bound_head(bd1)
        bound_logits = self._align(bound_logits, x)

        # Boundary attention map để inject vào seg decoder
        bnd_attn = self.bnd_attn_gate(bd1)  # [B, 1, D, H, W]

        # ─── Segmentation Decoder ───────────────────────────
        d4 = self.dec4(torch.cat([self._align(self.up4(b), e4),  self.ag4(self.up4(b), e4)], dim=1))
        d3 = self.dec3(torch.cat([self._align(self.up3(d4), e3), self.ag3(self.up3(d4), e3)], dim=1))
        d2 = self.dec2(torch.cat([self._align(self.up2(d3), e2), self.ag2(self.up2(d3), e2)], dim=1))
        d1 = self.dec1(torch.cat([self._align(self.up1(d2), e1), self.ag1(self.up1(d2), e1)], dim=1))

        # Inject boundary attention vào finest decoder feature
        d1_enhanced = d1 * (1.0 + self._align(bnd_attn, d1))

        seg_logits = self.seg_head(d1_enhanced)
        seg_logits = self._align(seg_logits, x)

        return seg_logits, bound_logits


# ---- Test Stage 2 ----
s2_model = BANet3D(
    in_channels=CFG["in_channels"],
    num_classes=CFG["num_classes"],
    base_features=CFG["s2_base_features"],
).to(DEVICE)

total = sum(p.numel() for p in s2_model.parameters())
print(f"[Stage 2] Parameters: {total:,}")

with torch.no_grad():
    dummy = torch.zeros(1, 1, 64, 64, 64).to(DEVICE)
    seg_out, bnd_out = s2_model(dummy)
    print(f"[Stage 2] Seg output:   {seg_out.shape}")
    print(f"[Stage 2] Bound output: {bnd_out.shape}")
del dummy; gc.collect()


## 6. Loss Functions

In [ ]:
# ============================================================
# LOSS FUNCTIONS
# ============================================================

# ─── Stage 1: Deep Supervision Loss ─────────────────────────
# Dice + CE tại mỗi scale, weighted sum (finest có weight cao nhất)

seg_loss_fn = DiceCELoss(
    include_background=False,
    to_onehot_y=True,
    softmax=True,
    lambda_dice=0.5,
    lambda_ce=0.5,
)


def deep_supervision_loss(outputs, label, ds_weights):
    """
    outputs: tuple (fine, ..., coarse) từ nnUNet khi training
    label  : [B, 1, D, H, W] long tensor
    ds_weights: list tương ứng, index 0 = finest
    """
    total = 0.0
    for i, (out, w) in enumerate(zip(outputs, ds_weights)):
        # Downsample label đến kích thước của output này
        if out.shape[2:] != label.shape[2:]:
            lbl_ds = F.interpolate(
                label.float(), size=out.shape[2:], mode="nearest"
            ).long()
        else:
            lbl_ds = label
        total += w * seg_loss_fn(out, lbl_ds)
    return total


# ─── Stage 2: Boundary Loss ──────────────────────────────────
bnd_loss_fn = nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor([10.0]).to(DEVICE)
)


def compute_boundary_gt(label_batch: torch.Tensor) -> torch.Tensor:
    """
    Tính ground-truth boundary map từ label.
    boundary = foreground XOR eroded_foreground
    label_batch: [B, 1, D, H, W] long
    returns: [B, 1, D, H, W] float
    """
    device = label_batch.device
    fg = (label_batch > 0).cpu().numpy().astype(np.uint8)
    boundaries = np.zeros_like(fg, dtype=np.float32)
    struct3d   = np.ones((3, 3, 3), dtype=bool)

    for b in range(fg.shape[0]):
        vol    = fg[b, 0]
        eroded = binary_erosion(vol, structure=struct3d).astype(np.uint8)
        boundaries[b, 0] = (vol - eroded).clip(0, 1)

    return torch.from_numpy(boundaries).to(device)


def total_loss_s2(seg_logits, bound_logits, label_batch, boundary_weight=0.4):
    """
    Stage 2 loss: (1-w)*seg_loss + w*boundary_loss
    """
    l_seg = seg_loss_fn(seg_logits, label_batch)
    bgt   = compute_boundary_gt(label_batch)
    l_bnd = bnd_loss_fn(bound_logits, bgt)
    total = (1 - boundary_weight) * l_seg + boundary_weight * l_bnd
    return total, l_seg.item(), l_bnd.item()


print("Loss functions sẵn sàng.")


## 7. Metrics

In [ ]:
# ============================================================
# METRICS: Dice, IoU, HD95
# ============================================================

dice_metric = DiceMetric(
    include_background=False,
    reduction="mean_batch",
    get_not_nans=True,
)

CLASS_NAMES = ["kidney", "tumor", "cyst"]  # class 1, 2, 3 (bỏ background)


def compute_metrics_batch(seg_logits, label_batch, num_classes):
    probs    = F.softmax(seg_logits, dim=1)
    preds_oh = one_hot(probs.argmax(dim=1, keepdim=True), num_classes=num_classes)
    lbls_oh  = one_hot(label_batch, num_classes=num_classes)
    dice_metric(y_pred=preds_oh, y=lbls_oh)


def aggregate_metrics():
    dice_vals, _ = dice_metric.aggregate()
    dice_metric.reset()

    results = {}
    dice_list = []
    for i, name in enumerate(CLASS_NAMES):
        d = dice_vals[i].item() if not torch.isnan(dice_vals[i]) else float("nan")
        results[f"dice_{name}"] = d
        results[f"iou_{name}"]  = d / (2.0 - d) if not np.isnan(d) else float("nan")
        dice_list.append(d)

    results["dice_mean"] = float(np.nanmean(dice_list))
    results["iou_mean"]  = float(np.nanmean([results[f"iou_{n}"] for n in CLASS_NAMES]))
    return results


def compute_hd95(pred_bin, gt_bin):
    """HD95 cho một class, input: [D,H,W] bool numpy arrays."""
    if pred_bin.sum() == 0 or gt_bin.sum() == 0:
        return float("nan")
    struct = np.ones((3,) * pred_bin.ndim, dtype=bool)
    e_pred = pred_bin ^ binary_erosion(pred_bin, structure=struct)
    e_gt   = gt_bin   ^ binary_erosion(gt_bin,   structure=struct)
    d1 = distance_transform_edt(~e_pred)[e_gt]
    d2 = distance_transform_edt(~e_gt)[e_pred]
    if len(d1) == 0 or len(d2) == 0:
        return float("nan")
    return float(np.percentile(np.concatenate([d1, d2]), 95))


print("Metrics sẵn sàng.")


## 8. Training — Stage 1 (nnU-Net Coarse)

In [ ]:
# ============================================================
# STAGE 1 TRAINING
# ============================================================

# ---- DataLoaders cho Stage 1 ----
s1_train_ds = KiTS23Dataset(
    train_files,
    transforms=make_train_transforms(CFG["s1_patch_size"], CFG["s1_num_patches"]),
    mode="stage1",
)
s1_val_ds = KiTS23Dataset(
    val_files,
    transforms=make_val_transforms(CFG["s1_patch_size"]),
    mode="stage1",
)

s1_train_loader = DataLoader(
    s1_train_ds, batch_size=CFG["s1_batch_size"], shuffle=True,
    num_workers=CFG["num_workers"], pin_memory=True,
    collate_fn=collate_patches, drop_last=True,
)
s1_val_loader = DataLoader(
    s1_val_ds, batch_size=1, shuffle=False,
    num_workers=CFG["num_workers"], pin_memory=True,
    collate_fn=collate_patches,
)

# ---- Optimizer & Scheduler ----
s1_optimizer = torch.optim.AdamW(
    s1_model.parameters(),
    lr=CFG["s1_lr"], weight_decay=CFG["s1_weight_decay"]
)

def lr_lambda_s1(epoch):
    warmup, total = CFG["s1_warmup_epochs"], CFG["s1_epochs"]
    if epoch < warmup:
        return (epoch + 1) / warmup
    return ((1 - (epoch - warmup) / max(1, total - warmup)) ** 0.9)

s1_scheduler = torch.optim.lr_scheduler.LambdaLR(s1_optimizer, lr_lambda_s1)
s1_scaler    = torch.amp.GradScaler("cuda", enabled=CFG["use_amp"])


def train_one_epoch_s1(model, loader, optimizer, scaler, device, ds_weights):
    model.train()
    total_loss_sum = 0.0
    steps = 0

    for batch in loader:
        imgs   = batch["image"].to(device, non_blocking=True)
        labels = batch["label"].to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast("cuda", enabled=CFG["use_amp"]):
            outputs = model(imgs)  # tuple (fine, ..., coarse)
            loss    = deep_supervision_loss(outputs, labels, ds_weights)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()

        total_loss_sum += loss.item()
        steps += 1

    return total_loss_sum / max(steps, 1)


@torch.no_grad()
def validate_s1(model, loader, device, num_classes):
    model.eval()
    val_loss_sum = 0.0
    steps = 0

    for batch in loader:
        imgs   = batch["image"].to(device, non_blocking=True)
        labels = batch["label"].to(device, non_blocking=True)

        with torch.amp.autocast("cuda", enabled=CFG["use_amp"]):
            logits = model(imgs)  # eval mode: chỉ trả về finest
            loss   = seg_loss_fn(logits, labels)

        compute_metrics_batch(logits, labels, num_classes)
        val_loss_sum += loss.item()
        steps += 1

    metrics = aggregate_metrics()
    metrics["val_loss"] = val_loss_sum / max(steps, 1)
    return metrics


# ---- Main Loop Stage 1 ----
s1_ckpt_path = os.path.join(CFG["output_dir"], "s1_checkpoints", "best_s1.pth")
s1_history   = []
best_s1_dice = -1.0
best_s1_epoch = -1

print(f"=" * 70)
print(f"[STAGE 1] Training nnU-Net — {CFG['s1_epochs']} epochs")
print(f"Train batches: {len(s1_train_loader)} | Val batches: {len(s1_val_loader)}")
print(f"=" * 70)

for epoch in range(1, CFG["s1_epochs"] + 1):
    t0 = time.time()
    train_loss = train_one_epoch_s1(
        s1_model, s1_train_loader, s1_optimizer, s1_scaler,
        DEVICE, CFG["s1_ds_weights"]
    )
    s1_scheduler.step()

    row = {"epoch": epoch, "lr": s1_optimizer.param_groups[0]["lr"], "train_loss": train_loss}

    if epoch % CFG["s1_val_every"] == 0 or epoch == CFG["s1_epochs"]:
        val_stats = validate_s1(s1_model, s1_val_loader, DEVICE, CFG["num_classes"])
        row.update(val_stats)

        if val_stats["dice_mean"] > best_s1_dice:
            best_s1_dice  = val_stats["dice_mean"]
            best_s1_epoch = epoch
            torch.save({
                "epoch": epoch, "model_state": s1_model.state_dict(),
                "optim_state": s1_optimizer.state_dict(),
                "best_metric": best_s1_dice, "cfg": CFG,
            }, s1_ckpt_path)

        print(
            f"[S1] Ep {epoch:03d}/{CFG['s1_epochs']} | "
            f"LR: {row['lr']:.2e} | Train: {train_loss:.4f} | "
            f"Val: {val_stats['val_loss']:.4f} | "
            f"Dice(mean): {val_stats['dice_mean']:.4f} "
            f"[kidney {val_stats['dice_kidney']:.3f} "
            f"tumor {val_stats['dice_tumor']:.3f} "
            f"cyst {val_stats['dice_cyst']:.3f}] | "
            f"Best: {best_s1_dice:.4f} (ep {best_s1_epoch}) | "
            f"{time.time()-t0:.0f}s"
        )
    else:
        print(f"[S1] Ep {epoch:03d}/{CFG['s1_epochs']} | LR: {row['lr']:.2e} | "
              f"Train: {train_loss:.4f} | {time.time()-t0:.0f}s")

    s1_history.append(row)

torch.save({"epoch": CFG["s1_epochs"], "model_state": s1_model.state_dict(), "cfg": CFG},
           os.path.join(CFG["output_dir"], "s1_checkpoints", "last_s1.pth"))
with open(os.path.join(CFG["output_dir"], "s1_history.json"), "w") as f:
    json.dump(s1_history, f, indent=2)

print(f"\n[STAGE 1] Xong. Best Dice: {best_s1_dice:.4f} tại epoch {best_s1_epoch}.")


## 9. Stage 1 → Stage 2 Bridge

Sau khi Stage 1 training xong:
1. Load best checkpoint
2. Chạy **sliding window inference** trên toàn bộ train/val cases
3. Lưu prediction `.pt` → Stage 2 sẽ dùng để crop ROI


In [ ]:
# ============================================================
# LOAD STAGE 1 CHECKPOINT TỪ KAGGLE VERSION CŨ
# Dùng cell này thay cho việc train lại Stage 1 từ đầu
#
# Cách dùng:
#   1. Vào notebook version đã train xong Stage 1
#   2. Tab Output → copy đường dẫn file best_s1.pth
#   3. Add version đó làm Input dataset của notebook này
#      (Edit notebook → + Add data → Your work → chọn version)
#   4. Điền đường dẫn vào S1_CKPT_EXTERNAL bên dưới
#   5. Chạy cell này, rồi bỏ qua cell "Training Stage 1"
#      và cell "Generate S1 predictions"
# ============================================================

import shutil

# Đường dẫn checkpoint từ version cũ (sau khi add làm Input)
# Ví dụ: /kaggle/input/nnunet-banet-kits23/output/s1_checkpoints/best_s1.pth
S1_CKPT_EXTERNAL = None  # ← điền đường dẫn vào đây

# Đường dẫn S1 predictions từ version cũ (nếu đã generate)
S1_PRED_EXTERNAL = None  # ← ví dụ: /kaggle/input/nnunet-banet-kits23/output/s1_predictions

if S1_CKPT_EXTERNAL is not None:
    # Copy checkpoint vào working dir
    dst = os.path.join(CFG['output_dir'], 's1_checkpoints', 'best_s1.pth')
    shutil.copy2(S1_CKPT_EXTERNAL, dst)
    print(f'Copied S1 checkpoint: {S1_CKPT_EXTERNAL} → {dst}')

    # Load vào model
    ckpt = torch.load(dst, map_location=DEVICE, weights_only=False)
    s1_model.load_state_dict(ckpt['model_state'])
    s1_model.eval()
    print(f"[S1] Loaded: epoch={ckpt['epoch']}, best_metric={ckpt['best_metric']:.4f}")
else:
    print('[INFO] S1_CKPT_EXTERNAL = None → bỏ qua, sẽ train Stage 1 từ đầu')

if S1_PRED_EXTERNAL is not None:
    # Copy toàn bộ predictions folder
    S1_PRED_DIR = os.path.join(CFG['output_dir'], 's1_predictions')
    if os.path.exists(S1_PRED_DIR):
        shutil.rmtree(S1_PRED_DIR)
    shutil.copytree(S1_PRED_EXTERNAL, S1_PRED_DIR)
    pred_files = glob.glob(os.path.join(S1_PRED_DIR, '*.pt'))
    print(f'Copied {len(pred_files)} S1 predictions → {S1_PRED_DIR}')
    print('[INFO] Có thể bỏ qua cell Generate S1 predictions')
else:
    print('[INFO] S1_PRED_EXTERNAL = None → cần chạy cell Generate S1 predictions')


In [ ]:
import warnings
# Warning này đến từ nội bộ MONAI sliding_window_inference,
# không ảnh hưởng kết quả — suppress để log sạch hơn
warnings.filterwarnings(
    "ignore",
    message="Using a non-tuple sequence for multidimensional indexing",
    category=UserWarning,
)

# ============================================================
# STAGE 1 INFERENCE → Tạo ROI crops cho Stage 2
# ============================================================

S1_PRED_DIR = os.path.join(CFG["output_dir"], "s1_predictions")
os.makedirs(S1_PRED_DIR, exist_ok=True)

# Load best Stage 1 checkpoint
s1_ckpt = torch.load(s1_ckpt_path, map_location=DEVICE, weights_only=False)
s1_model.load_state_dict(s1_ckpt["model_state"])
s1_model.eval()
print(f"[S1] Loaded best checkpoint (epoch {s1_ckpt['epoch']}, "
      f"Dice={s1_ckpt['best_metric']:.4f})")


@torch.no_grad()
def generate_s1_predictions(model, file_list, device, patch_size, pred_dir, overlap=0.5):
    """
    Chạy Stage 1 sliding window inference và lưu prediction.
    Predictions dùng làm ROI guide cho Stage 2 training.
    """
    model.eval()
    for i, entry in enumerate(file_list, 1):
        case_id = os.path.splitext(os.path.basename(entry["image"]))[0]
        out_path = os.path.join(pred_dir, f"{case_id}_s1pred.pt")

        if os.path.exists(out_path):
            print(f"[{i}/{len(file_list)}] {case_id}: đã có prediction, bỏ qua")
            continue

        img = torch.load(entry["image"], weights_only=False).float().unsqueeze(0).to(device)

        with torch.amp.autocast("cuda", enabled=CFG["use_amp"]):
            logits = sliding_window_inference(
                img, roi_size=patch_size, sw_batch_size=2,
                predictor=model, overlap=overlap, mode="gaussian",
            )

        pred = logits.argmax(dim=1, keepdim=True).cpu().to(torch.uint8)  # [1, 1, D, H, W]
        torch.save(pred, out_path)
        print(f"[{i}/{len(file_list)}] {case_id}: saved → {out_path}")

        del img, logits, pred
        gc.collect()
        torch.cuda.empty_cache()


# Tạo S1 predictions cho train + val (Stage 2 cần để crop ROI)
all_train_val = train_files + val_files
print(f"\nGenerating Stage 1 predictions cho {len(all_train_val)} cases...")
generate_s1_predictions(
    s1_model, all_train_val, DEVICE, CFG["s1_patch_size"], S1_PRED_DIR
)
print("Hoàn thành! Stage 1 predictions sẵn sàng cho Stage 2.")


## 10. Training — Stage 2 (BA-Net Fine)

In [ ]:
# ============================================================
# STAGE 2 TRAINING
# ============================================================

# ---- DataLoaders cho Stage 2 ----
# mode='stage2': tự động crop ROI từ S1 prediction trước khi apply transforms
s2_train_ds = KiTS23Dataset(
    train_files,
    transforms=make_train_transforms(CFG["s2_patch_size"], CFG["s2_num_patches"]),
    mode="stage2",
    stage1_pred_dir=S1_PRED_DIR,
    crop_margin=CFG["s2_crop_margin"],
)
s2_val_ds = KiTS23Dataset(
    val_files,
    transforms=make_val_transforms(CFG["s2_patch_size"]),
    mode="stage2",
    stage1_pred_dir=S1_PRED_DIR,
    crop_margin=CFG["s2_crop_margin"],
)

s2_train_loader = DataLoader(
    s2_train_ds, batch_size=CFG["s2_batch_size"], shuffle=True,
    num_workers=CFG["num_workers"], pin_memory=True,
    collate_fn=collate_patches, drop_last=True,
)
s2_val_loader = DataLoader(
    s2_val_ds, batch_size=1, shuffle=False,
    num_workers=CFG["num_workers"], pin_memory=True,
    collate_fn=collate_patches,
)

# ---- Optimizer & Scheduler ----
s2_optimizer = torch.optim.AdamW(
    s2_model.parameters(),
    lr=CFG["s2_lr"], weight_decay=CFG["s2_weight_decay"]
)

def lr_lambda_s2(epoch):
    warmup, total = CFG["s2_warmup_epochs"], CFG["s2_epochs"]
    if epoch < warmup:
        return (epoch + 1) / warmup
    return ((1 - (epoch - warmup) / max(1, total - warmup)) ** 0.9)

s2_scheduler = torch.optim.lr_scheduler.LambdaLR(s2_optimizer, lr_lambda_s2)
s2_scaler    = torch.amp.GradScaler("cuda", enabled=CFG["use_amp"])


def train_one_epoch_s2(model, loader, optimizer, scaler, device, bw):
    model.train()
    sum_loss = sum_seg = sum_bnd = 0.0
    steps = 0

    for batch in loader:
        imgs   = batch["image"].to(device, non_blocking=True)
        labels = batch["label"].to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast("cuda", enabled=CFG["use_amp"]):
            seg_logits, bound_logits = model(imgs)
            loss, l_seg, l_bnd = total_loss_s2(seg_logits, bound_logits, labels, bw)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()

        sum_loss += loss.item()
        sum_seg  += l_seg
        sum_bnd  += l_bnd
        steps    += 1

    n = max(steps, 1)
    return {"loss": sum_loss/n, "seg_loss": sum_seg/n, "bnd_loss": sum_bnd/n}


@torch.no_grad()
def validate_s2(model, loader, device, num_classes, bw):
    model.eval()
    sum_loss = 0.0
    steps = 0

    for batch in loader:
        imgs   = batch["image"].to(device, non_blocking=True)
        labels = batch["label"].to(device, non_blocking=True)

        with torch.amp.autocast("cuda", enabled=CFG["use_amp"]):
            seg_logits, bound_logits = model(imgs)
            loss, _, _ = total_loss_s2(seg_logits, bound_logits, labels, bw)

        compute_metrics_batch(seg_logits, labels, num_classes)
        sum_loss += loss.item()
        steps    += 1

    metrics = aggregate_metrics()
    metrics["val_loss"] = sum_loss / max(steps, 1)
    return metrics


# ---- Main Loop Stage 2 ----
s2_ckpt_path  = os.path.join(CFG["output_dir"], "s2_checkpoints", "best_s2.pth")
s2_history    = []
best_s2_dice  = -1.0
best_s2_epoch = -1

print(f"=" * 70)
print(f"[STAGE 2] Training BA-Net — {CFG['s2_epochs']} epochs")
print(f"Train batches: {len(s2_train_loader)} | Val batches: {len(s2_val_loader)}")
print(f"Boundary weight: {CFG['s2_boundary_weight']}")
print(f"=" * 70)

for epoch in range(1, CFG["s2_epochs"] + 1):
    t0 = time.time()
    train_stats = train_one_epoch_s2(
        s2_model, s2_train_loader, s2_optimizer, s2_scaler,
        DEVICE, CFG["s2_boundary_weight"]
    )
    s2_scheduler.step()

    row = {
        "epoch": epoch,
        "lr": s2_optimizer.param_groups[0]["lr"],
        **{f"train_{k}": v for k, v in train_stats.items()},
    }

    if epoch % CFG["s2_val_every"] == 0 or epoch == CFG["s2_epochs"]:
        val_stats = validate_s2(
            s2_model, s2_val_loader, DEVICE,
            CFG["num_classes"], CFG["s2_boundary_weight"]
        )
        row.update({f"val_{k}": v for k, v in val_stats.items()})

        if val_stats["dice_mean"] > best_s2_dice:
            best_s2_dice  = val_stats["dice_mean"]
            best_s2_epoch = epoch
            torch.save({
                "epoch": epoch, "model_state": s2_model.state_dict(),
                "optim_state": s2_optimizer.state_dict(),
                "best_metric": best_s2_dice, "cfg": CFG,
            }, s2_ckpt_path)

        print(
            f"[S2] Ep {epoch:03d}/{CFG['s2_epochs']} | "
            f"LR: {row['lr']:.2e} | "
            f"Train: {row['train_loss']:.4f} "
            f"(seg {row['train_seg_loss']:.4f} bnd {row['train_bnd_loss']:.4f}) | "
            f"Val: {val_stats['val_loss']:.4f} | "
            f"Dice(mean): {val_stats['dice_mean']:.4f} "
            f"[kidney {val_stats['dice_kidney']:.3f} "
            f"tumor {val_stats['dice_tumor']:.3f} "
            f"cyst {val_stats['dice_cyst']:.3f}] | "
            f"Best: {best_s2_dice:.4f} (ep {best_s2_epoch}) | "
            f"{time.time()-t0:.0f}s"
        )
    else:
        print(
            f"[S2] Ep {epoch:03d}/{CFG['s2_epochs']} | LR: {row['lr']:.2e} | "
            f"Train: {row['train_loss']:.4f} | {time.time()-t0:.0f}s"
        )

    s2_history.append(row)

torch.save({"epoch": CFG["s2_epochs"], "model_state": s2_model.state_dict(), "cfg": CFG},
           os.path.join(CFG["output_dir"], "s2_checkpoints", "last_s2.pth"))
with open(os.path.join(CFG["output_dir"], "s2_history.json"), "w") as f:
    json.dump(s2_history, f, indent=2)

print(f"\n[STAGE 2] Xong. Best Dice: {best_s2_dice:.4f} tại epoch {best_s2_epoch}.")


## 11. Training Curves

In [ ]:
import matplotlib.pyplot as plt


def plot_training_curves(history_path, title, save_path, stage="s2"):
    with open(history_path) as f:
        history = json.load(f)

    epochs      = [r["epoch"] for r in history]
    train_loss  = [r["train_loss"] for r in history]
    val_rows    = [r for r in history if "val_dice_mean" in r]
    val_epochs  = [r["epoch"] for r in val_rows]
    val_loss    = [r["val_val_loss"] for r in val_rows]
    val_dice_m  = [r["val_dice_mean"] for r in val_rows]
    val_dice_k  = [r["val_dice_kidney"] for r in val_rows]
    val_dice_tu = [r.get("val_dice_tumor", r.get("val_dice_mass", 0)) for r in val_rows]
    val_dice_cy = [r["val_dice_cyst"] for r in val_rows]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(title, fontsize=14)

    axes[0].plot(epochs, train_loss, label="Train loss", color="steelblue")
    axes[0].plot(val_epochs, val_loss, label="Val loss", color="tomato", marker="o")
    axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss")
    axes[0].set_title("Loss"); axes[0].legend(); axes[0].grid(True, alpha=0.3)

    axes[1].plot(val_epochs, val_dice_m,  label="Mean",   color="black",    lw=2, marker="o")
    axes[1].plot(val_epochs, val_dice_k,  label="Kidney", color="steelblue", ls="--")
    axes[1].plot(val_epochs, val_dice_tu, label="Tumor",  color="tomato",    ls="--")
    axes[1].plot(val_epochs, val_dice_cy, label="Cyst",   color="seagreen",  ls="--")
    axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Dice")
    axes[1].set_title("Validation Dice"); axes[1].legend()
    axes[1].grid(True, alpha=0.3); axes[1].set_ylim([0, 1])

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {save_path}")


# Stage 1 curves (chỉ train_loss vì val keys khác)
with open(os.path.join(CFG["output_dir"], "s1_history.json")) as f:
    s1_hist = json.load(f)

s1_train_epochs = [r["epoch"] for r in s1_hist]
s1_train_losses = [r["train_loss"] for r in s1_hist]
s1_val_rows     = [r for r in s1_hist if "dice_mean" in r]

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Stage 1 — nnU-Net Training", fontsize=14)
ax[0].plot(s1_train_epochs, s1_train_losses, color="steelblue", label="Train loss")
if s1_val_rows:
    ax[0].plot([r["epoch"] for r in s1_val_rows], [r["val_loss"] for r in s1_val_rows],
               color="tomato", marker="o", label="Val loss")
ax[0].set_title("Loss"); ax[0].legend(); ax[0].grid(True, alpha=0.3)
if s1_val_rows:
    ax[1].plot([r["epoch"] for r in s1_val_rows], [r["dice_mean"] for r in s1_val_rows],
               color="black", lw=2, marker="o", label="Mean Dice")
    ax[1].set_title("Validation Dice"); ax[1].legend(); ax[1].grid(True, alpha=0.3); ax[1].set_ylim([0,1])
plt.tight_layout()
plt.savefig(os.path.join(CFG["output_dir"], "s1_curves.png"), dpi=150, bbox_inches="tight")
plt.show()

# Stage 2 curves
plot_training_curves(
    os.path.join(CFG["output_dir"], "s2_history.json"),
    "Stage 2 — BA-Net Training",
    os.path.join(CFG["output_dir"], "s2_curves.png"),
)


## 12. Evaluation trên Test Set

In [ ]:
# ============================================================
# FINAL EVALUATION — 2-Stage Pipeline trên Test Set
#
# Inference flow:
#   1. Stage 1: sliding window trên full volume → coarse mask
#   2. Tính bounding box từ coarse mask + margin
#   3. Stage 2: sliding window trên ROI crop → fine mask
#   4. Paste prediction về original space
#   5. Tính Dice, IoU, HD95
# ============================================================

import nibabel as nib
import pandas as pd

# Load best checkpoints
s1_ckpt = torch.load(s1_ckpt_path, map_location=DEVICE, weights_only=False)
s1_model.load_state_dict(s1_ckpt["model_state"])
s1_model.eval()

s2_ckpt = torch.load(s2_ckpt_path, map_location=DEVICE, weights_only=False)
s2_model.load_state_dict(s2_ckpt["model_state"])
s2_model.eval()

print(f"[S1] epoch {s1_ckpt['epoch']}, Dice={s1_ckpt['best_metric']:.4f}")
print(f"[S2] epoch {s2_ckpt['epoch']}, Dice={s2_ckpt['best_metric']:.4f}")


def get_roi_bbox(mask_tensor, margin):
    """
    mask_tensor: [1, D, H, W] int
    returns: (d0,d1,h0,h1,w0,w1)
    """
    fg = (mask_tensor.squeeze(0) > 0)
    if fg.sum() == 0:
        return None
    coords = torch.nonzero(fg, as_tuple=False)
    D, H, W = fg.shape
    m = margin
    return (
        max(coords[:,0].min().item()-m, 0), min(coords[:,0].max().item()+m+1, D),
        max(coords[:,1].min().item()-m, 0), min(coords[:,1].max().item()+m+1, H),
        max(coords[:,2].min().item()-m, 0), min(coords[:,2].max().item()+m+1, W),
    )


@torch.no_grad()
def inference_two_stage(img_tensor, s1_model, s2_model, device,
                         s1_patch, s2_patch, crop_margin, num_classes):
    """
    Full 2-stage inference trên 1 volume.
    img_tensor: [1, D, H, W] float — đã preprocessed
    returns: pred [D, H, W] int (argmax)
    """
    img = img_tensor.unsqueeze(0).to(device)  # [1, 1, D, H, W]
    D, H, W = img.shape[2], img.shape[3], img.shape[4]

    # --- Stage 1: coarse segmentation ---
    with torch.amp.autocast("cuda", enabled=CFG["use_amp"]):
        s1_logits = sliding_window_inference(
            img, roi_size=s1_patch, sw_batch_size=2,
            predictor=s1_model, overlap=0.5, mode="gaussian",
        )
    s1_pred = s1_logits.argmax(dim=1, keepdim=True)  # [1, 1, D, H, W]

    # --- Crop ROI từ Stage 1 ---
    bbox = get_roi_bbox(s1_pred.squeeze(0).cpu(), crop_margin)
    if bbox is None:
        # Không có foreground → trả về Stage 1 prediction
        return s1_pred.squeeze().cpu().numpy().astype(np.uint8)

    d0,d1,h0,h1,w0,w1 = bbox
    img_crop = img[:, :, d0:d1, h0:h1, w0:w1]  # [1, 1, rd, rh, rw]

    # --- Stage 2: fine segmentation trên crop ---
    with torch.amp.autocast("cuda", enabled=CFG["use_amp"]):
        s2_logits, _ = sliding_window_inference(
            img_crop, roi_size=s2_patch, sw_batch_size=2,
            predictor=lambda x: s2_model(x),  # trả về (seg, bnd)
            overlap=0.5, mode="gaussian",
        ) if False else (
            # sliding_window_inference chỉ nhận 1 output, cần wrapper
            sliding_window_inference(
                img_crop, roi_size=s2_patch, sw_batch_size=2,
                predictor=lambda x: s2_model(x)[0],
                overlap=0.5, mode="gaussian",
            ),
            None
        )

    s2_pred_crop = s2_logits.argmax(dim=1).squeeze(0).cpu()  # [rd, rh, rw]

    # --- Paste về original space ---
    final_pred = torch.zeros(D, H, W, dtype=torch.long)
    final_pred[d0:d1, h0:h1, w0:w1] = s2_pred_crop

    return final_pred.numpy().astype(np.uint8)


@torch.no_grad()
def evaluate_test_set(s1_model, s2_model, test_files, device, num_classes):
    all_results = []

    for entry in test_files:
        case_id = os.path.splitext(os.path.basename(entry["image"]))[0]

        img = torch.load(entry["image"], weights_only=False).float()   # [1, D, H, W]
        lbl = torch.load(entry["label"], weights_only=False).long()    # [1, D, H, W]

        pred_np = inference_two_stage(
            img, s1_model, s2_model, device,
            CFG["s1_patch_size"], CFG["s2_patch_size"],
            CFG["s2_crop_margin"], num_classes,
        )  # [D, H, W]

        pred_t = torch.from_numpy(pred_np).long().unsqueeze(0).unsqueeze(0)  # [1,1,D,H,W]
        lbl_t  = lbl.unsqueeze(0)  # [1, 1, D, H, W]

        # Dice metrics
        seg_logits_fake = F.one_hot(pred_t.squeeze(1), num_classes).permute(0,4,1,2,3).float()
        compute_metrics_batch(seg_logits_fake, lbl_t, num_classes)
        metrics = aggregate_metrics()

        # IoU from Dice
        for name in CLASS_NAMES:
            d = metrics[f"dice_{name}"]
            metrics[f"iou_{name}"] = d / (2 - d) if not np.isnan(d) else float("nan")

        # HD95
        lbl_np   = lbl.squeeze().numpy()
        hd95_list = []
        for i, name in enumerate(CLASS_NAMES):
            c   = i + 1
            hd  = compute_hd95(pred_np == c, lbl_np == c)
            metrics[f"hd95_{name}"] = hd
            hd95_list.append(hd)
        metrics["hd95_mean"] = float(np.nanmean(hd95_list))
        metrics["case_id"]   = case_id

        all_results.append(metrics)

        # Save NIfTI
        nii = nib.Nifti1Image(pred_np, affine=np.eye(4))
        nib.save(nii, os.path.join(CFG["output_dir"], "nifti_predictions", f"{case_id}_pred.nii.gz"))

        print(
            f"{case_id} | "
            f"Dice: kidney={metrics['dice_kidney']:.3f} "
            f"tumor={metrics['dice_tumor']:.3f} "
            f"cyst={metrics['dice_cyst']:.3f} "
            f"mean={metrics['dice_mean']:.3f} | "
            f"HD95: kidney={metrics['hd95_kidney']:.1f} "
            f"tumor={metrics['hd95_tumor']:.1f} "
            f"cyst={metrics['hd95_cyst']:.1f}mm"
        )

        del img, lbl
        gc.collect(); torch.cuda.empty_cache()

    return all_results


print("\n=" * 70)
print("FINAL EVALUATION — 2-Stage Pipeline trên Test Set")
print("=" * 70)
test_results = evaluate_test_set(
    s1_model, s2_model, test_files, DEVICE, CFG["num_classes"]
)


## 13. Kết quả & Visualization

In [ ]:
# ============================================================
# TỔNG KẾT KẾT QUẢ
# ============================================================

results_df = pd.DataFrame(test_results).set_index("case_id")
summary    = results_df.agg(["mean", "std", "min", "max"])

cols = [
    "dice_kidney",  "dice_tumor",  "dice_cyst",  "dice_mean",
    "iou_kidney",   "iou_tumor",   "iou_cyst",   "iou_mean",
    "hd95_kidney",  "hd95_tumor",  "hd95_cyst",  "hd95_mean",
]

print("\n=" * 70)
print("TEST SET RESULTS SUMMARY")
print("=" * 70)
print(summary[cols].round(4).to_string())

results_df.to_csv(os.path.join(CFG["output_dir"], "test_results_per_case.csv"))
summary.to_csv(os.path.join(CFG["output_dir"], "test_results_summary.csv"))

# ---- Box plots ----
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("Test Set Metrics — 2-Stage Pipeline (nnU-Net + BA-Net)", fontsize=13)

results_df[["dice_kidney", "dice_tumor", "dice_cyst"]].boxplot(ax=axes[0])
axes[0].set_title("Dice Score"); axes[0].set_ylim([0, 1])
axes[0].set_xticklabels(["Kidney", "Tumor", "Cyst"]); axes[0].grid(True, alpha=0.3)

results_df[["iou_kidney", "iou_tumor", "iou_cyst"]].boxplot(ax=axes[1])
axes[1].set_title("IoU Score"); axes[1].set_ylim([0, 1])
axes[1].set_xticklabels(["Kidney", "Tumor", "Cyst"]); axes[1].grid(True, alpha=0.3)

results_df[["hd95_kidney", "hd95_tumor", "hd95_cyst"]].boxplot(ax=axes[2])
axes[2].set_title("HD95 (mm) ↓"); axes[2].set_xticklabels(["Kidney", "Tumor", "Cyst"])
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(CFG["output_dir"], "test_metrics_boxplot.png"), dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# ============================================================
# VISUALIZATION — So sánh CT / GT / Stage1 / Stage2
# ============================================================

@torch.no_grad()
def visualize_two_stage(entry, s1_model, s2_model, device, num_slices=4):
    case_id = os.path.splitext(os.path.basename(entry["image"]))[0]
    img = torch.load(entry["image"], weights_only=False).float()
    lbl = torch.load(entry["label"], weights_only=False).long()

    # Stage 1 prediction
    img_dev = img.unsqueeze(0).to(device)
    with torch.amp.autocast("cuda", enabled=CFG["use_amp"]):
        s1_logits = sliding_window_inference(
            img_dev, roi_size=CFG["s1_patch_size"], sw_batch_size=2,
            predictor=s1_model, overlap=0.5, mode="gaussian",
        )
    s1_pred_np = s1_logits.argmax(dim=1).squeeze().cpu().numpy()

    # Stage 2 prediction (2-stage)
    pred_np = inference_two_stage(
        img, s1_model, s2_model, device,
        CFG["s1_patch_size"], CFG["s2_patch_size"],
        CFG["s2_crop_margin"], CFG["num_classes"],
    )

    img_np = img.squeeze().numpy()
    lbl_np = lbl.squeeze().numpy()

    # Chọn slices có foreground
    fg_slices = np.where(lbl_np.sum(axis=(1, 2)) > 50)[0]
    if len(fg_slices) < num_slices:
        slice_idx = np.linspace(0, lbl_np.shape[0]-1, num_slices, dtype=int)
    else:
        step = len(fg_slices) // num_slices
        slice_idx = fg_slices[::step][:num_slices]

    cmap = plt.colormaps["tab10"].resampled(4)
    fig, axes = plt.subplots(num_slices, 4, figsize=(16, 4 * num_slices))
    fig.suptitle(f"{case_id} — CT | Ground Truth | Stage1 (nnU-Net) | Stage2 (BA-Net)", fontsize=12)

    for row, s in enumerate(slice_idx):
        for col, (data, title) in enumerate([
            (None, f"CT slice {s}"),
            (lbl_np[s],     "Ground Truth"),
            (s1_pred_np[s], "Stage 1 (nnU-Net)"),
            (pred_np[s],    "Stage 2 (BA-Net)"),
        ]):
            ax = axes[row, col] if num_slices > 1 else axes[col]
            ax.imshow(img_np[s], cmap="gray")
            if data is not None:
                ax.imshow(data, cmap=cmap, alpha=0.5, vmin=0, vmax=3)
            ax.set_title(title); ax.axis("off")

    plt.tight_layout()
    save_path = os.path.join(CFG["output_dir"], f"viz_{case_id}.png")
    plt.savefig(save_path, dpi=120, bbox_inches="tight")
    plt.show()
    print(f"Saved: {save_path}")

    del img_dev, s1_logits
    gc.collect(); torch.cuda.empty_cache()


# Visualize 3 test cases đầu tiên
for entry in test_files[:3]:
    visualize_two_stage(entry, s1_model, s2_model, DEVICE)


## 14. Output Summary

Kiểm tra tất cả file output đã được tạo:


In [ ]:
# ============================================================
# KIỂM TRA OUTPUT
# ============================================================

output_files = [
    ("Stage 1 best ckpt",    os.path.join(CFG["output_dir"], "s1_checkpoints", "best_s1.pth")),
    ("Stage 2 best ckpt",    os.path.join(CFG["output_dir"], "s2_checkpoints", "best_s2.pth")),
    ("Stage 1 last ckpt",    os.path.join(CFG["output_dir"], "s1_checkpoints", "last_s1.pth")),
    ("Stage 2 last ckpt",    os.path.join(CFG["output_dir"], "s2_checkpoints", "last_s2.pth")),
    ("S1 training history",  os.path.join(CFG["output_dir"], "s1_history.json")),
    ("S2 training history",  os.path.join(CFG["output_dir"], "s2_history.json")),
    ("Test results CSV",     os.path.join(CFG["output_dir"], "test_results_summary.csv")),
    ("S1 training curves",   os.path.join(CFG["output_dir"], "s1_curves.png")),
    ("S2 training curves",   os.path.join(CFG["output_dir"], "s2_curves.png")),
    ("Test metrics boxplot", os.path.join(CFG["output_dir"], "test_metrics_boxplot.png")),
    ("Data split JSON",      os.path.join(CFG["output_dir"], "data_split.json")),
]

print("=" * 65)
print(f"{'STATUS':<8} {'SIZE':>8}   DESCRIPTION")
print("=" * 65)
for desc, path in output_files:
    exists = os.path.exists(path)
    size   = os.path.getsize(path) / 1e6 if exists else 0
    status = "✅ OK" if exists else "❌ MISS"
    print(f"{status:<8} {size:>6.1f}MB   {desc}")

# NIfTI predictions
nifti_preds = glob.glob(os.path.join(CFG["output_dir"], "nifti_predictions", "*.nii.gz"))
print(f"\nNIfTI predictions: {len(nifti_preds)} files")
print("=" * 65)
print("Pipeline hoàn thành!")
